# Mathematical framework and verification

Here we review the mathematical framework and verify and record the precise computations, assumptions, derivations, analytically predictable examples and small executable checks underlying the token-counting experiment (for bookeeping and reproducibility).

**References.** Zhang, Lipton, Li & Smola, *Dive into Deep Learning*, §10.2 (GRU); Cho et al. (2014), *Learning Phrase Representations using RNN Encoder–Decoder for Statistical Machine Translation*; and [PyTorch `nn.GRU` reference](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html).

## Notation conventions and execution contract

| Symbol | Meaning | Shape or range |
|:--|:--|:--|
| $T$ | Number of timesteps in one batch | integer $\geq 1$ for generated data |
| $B$ | Number of sequences in a batch | integer $\geq 1$ |
| $D$ | Input features per token | positive integer |
| $K$ | Number of GRU hidden coordinates | positive integer |
| $X$ | Full input tensor | $\mathbb R^{T\times B\times D}$ |
| $X_t$ | One timestep of inputs | $\mathbb R^{B\times D}$ |
| $H_0,H_t$ | Initial and subsequent states | $\mathbb R^{B\times K}$ |
| $R_t,Z_t,\widetilde H_t$ | Reset gate, update gate, candidate | $\mathbb R^{B\times K}$ |
| $C_{t,b}$ | Prefix count through step $t$ for sequence $b$ | $\{0,\ldots,t\}$ |
| $Y_b=C_{T,b}$ | Terminal target count | $\{0,\ldots,T\}$ |
| $\hat y_b$ | Continuous predicted count | $\mathbb R$ |
| $p$ | Probability a token is one | $[0,1]$ |

We use **time-major, row-oriented batches**.

Computations start at $t=1$; Python iteration starts at index `0`.

**Notation convention:** capital letters denote batched tensors, while lowercase $x_{t,b}$, $c_{t,b}$, $y_b$, and $\hat y_b$ denote values for a single sequence. Matrix multiplication is written for **row vectors**: $H_TW_{\mathrm{out}}^\top+b_{\mathrm{out}}$, not $W_{\mathrm{out}}H_T$.

## 1. Recurrent computation
Baseline model implementation follows Zhang et al.

**Implementation:** `GRUScratch.__init__()` and `GRUScratch.forward()`.

For input dimension $D$ and hidden dimension $K$:

- Inputs: $X\in\mathbb{R}^{T\times B\times D}$.
- Hidden state: $H_t\in\mathbb{R}^{B\times K}$.
- Input weights: $W_{x*}\in\mathbb{R}^{D\times K}$.
- Recurrent weights: $W_{h*}\in\mathbb{R}^{K\times K}$.

The recurrent transition $H_t=F_\theta(X_t,H_{t-1})$ is

$$
\begin{aligned}
Z_t &= \operatorname{sigmoid}(X_tW_{xz}+H_{t-1}W_{hz}+b_z),\\
R_t &= \operatorname{sigmoid}(X_tW_{xr}+H_{t-1}W_{hr}+b_r),\\
\widetilde H_t &= \tanh\!\left(X_tW_{xh}+(R_t\odot H_{t-1})W_{hh}+b_h\right),\\
H_t &= Z_t\odot H_{t-1}+(1-Z_t)\odot\widetilde H_t.
\end{aligned}
$$

Here $\odot$ denotes elementwise multiplication and the activation functions act elementwise. Each gate uses **independent** parameters. In our convention **$Z_t\approx1$ retains the old state**, whereas $Z_t\approx0$ replaces it with the candidate. The reset gate instead determines which components of the previous state enter the *candidate* computation; it does not directly reset the final state.

**Shapes.** For $g\in\{z,r,h\}$, $W_{xg}\in\mathbb R^{D\times K}$, $W_{hg}\in\mathbb R^{K\times K}$ and $b_g\in\mathbb R^K$. Consequently every gate, candidate and updated state has shape $(B,K)$.

**Reference distinction.** Our candidate multiplies **$(R_t\odot H_{t-1})W_{hh}$**. In PyTorch, native `nn.GRU` uses a candidate of the form

$$n_t = \tanh\left(X_t W_{in\_n} + b_{in\_n} + R_t \odot (H_{t-1} W_{hn\_n} + b_{hn\_n})\right)$$

applying the reset gate after the recurrent linear transformation. The two reset placements are not equivalent. Direct numerical equality with native `nn.GRU` is *not* the correctness criterion for this implementation.

### 1.1 Constructor, random initialisation and parameter count

**Computation.** `GRUScratch(D,K,sigma)` registers three parameter triples $(W_{xg},W_{hg},b_g)$, for update, reset and candidate transformations. Each weight element is drawn as $W_{ij}=\sigma\epsilon_{ij}$ with $\epsilon_{ij}\sim\mathcal N(0,1)$, hence $W_{ij}\sim\mathcal N(0,\sigma^2)$. Each bias starts at zero. The default is $\sigma=0.01$.

**Code correspondence (`gru_vanilla.py`, lines 20–33):**

```python
return nn.Parameter(torch.randn(*shape) * sigma)
# Each of three calls to triple() creates (D,K) weights, (K,K) weights, (K,) bias.
```

**Parameter count derivation.** One triple contains $DK+K^2+K$ trainable scalars, so

$$N_{\mathrm{GRU}}=3(DK+K^2+K).$$

For $D=3,K=5$: $N_{\mathrm{GRU}}=3(15+25+5)=135$. For the counting architecture's $D=1,K=16$: $N_{\mathrm{GRU}}=3(16+256+16)=864$.

**Assumptions and limitations.** `sigma` is the **standard deviation**, not variance. The code uses the global PyTorch RNG state unless seeded by its caller; initialisation does not establish that $\sigma=0.01$ is optimal for learning. The six weight matrices and three biases are separate trainable tensors.

**Independent edge case.** When $\sigma=0$, all weights must be exactly zero irrespective of the random draws. This is a deterministic implementation check.

In [11]:
import math
import torch

from gru_vanilla import GRUScratch
from counting_experiment import (
    CountingGRUSystem,
    _model_input,
    generate_counting_batch,
    length_batches,
)

# Constructor and registration checks, including a deterministic sigma=0 case.
for D, K, expected in ((3, 5, 135), (1, 16, 864)):
    torch.manual_seed(11)
    gru = GRUScratch(D, K)
    params = dict(gru.named_parameters())
    assert len(params) == 9
    assert sum(p.numel() for p in params.values()) == expected
    assert all(p.requires_grad for p in params.values())
    for gate in ("z", "r", "h"):
        assert params[f"W_x{gate}"].shape == (D, K)
        assert params[f"W_h{gate}"].shape == (K, K)
        assert params[f"b_{gate}"].shape == (K,)
        assert torch.count_nonzero(params[f"b_{gate}"]).item() == 0

zero_gru = GRUScratch(1, 2, sigma=0)
assert all(torch.count_nonzero(p).item() == 0 for p in zero_gru.parameters())
print("PASS — registered shapes, 135/864 parameter counts, trainability, zero biases, sigma=0")

PASS — registered shapes, 135/864 parameter counts, trainability, zero biases, sigma=0


### 1.2 Forward pass: initial state, gate coefficients, and recurrent update

**Computation and shapes.** `forward(inputs, H=None, ...)` consumes $X\in\mathbb R^{T\times B\times D}$; if `H` is omitted it sets $H_0=0\in\mathbb R^{B\times K}$ using the input dtype/device. At each timestep the GRU computes $Z_t$, $R_t$, $\widetilde H_t$, then $H_t$, storing each $H_t$. The ordinary return is `([H_1, ..., H_T], H_T)`.

**Code correspondence (`gru_vanilla.py`, lines 43–71):**

```python
if H is None:
    H = inputs.new_zeros((inputs.shape[1], self.num_hiddens))
for t, X in enumerate(inputs):
    # Compute Z, R, H_tilde from the four equations above.
    H = Z_effective * H + (1 - Z_effective) * H_tilde
    outputs.append(H)
```

**Why the update is an interpolation.** For each coordinate, $h_t=zh_{t-1}+(1-z)\widetilde h_t$, where $0\leq z\leq1$ in exact arithmetic. Hence:

- $z=1$: retain $h_{t-1}$;
- $z=0$: replace with $\widetilde h_t$;
- $z=1/2$: average the two.

**Analytical two-step example.** Take $B=D=K=1$, $H_0=0.4$, $x_1=1$, $x_2=0$. Let gate preactivations be zero, so $z_t=r_t=1/2$, and set $W_{xh}=0.8$, $W_{hh}=0.3$, $b_h=0$. Then

$$\widetilde h_t=\tanh(0.8x_t+0.15h_{t-1}),\qquad h_t=\tfrac12h_{t-1}+\tfrac12\widetilde h_t.$$

Thus

$$h_1=0.2+\tfrac12\tanh(0.86),\qquad h_2=\tfrac12h_1+\tfrac12\tanh(0.15h_1).$$

The test evaluates these scalar expressions with Python's `math.tanh` rather than implementing a second PyTorch GRU.

In [12]:
# Evaluate scalar, two-step recurrence
gru = GRUScratch(1, 1, sigma=0).double()
with torch.no_grad():
    gru.W_xh.fill_(0.8)
    gru.W_hh.fill_(0.3)

sequence = torch.tensor([1.0, 0.0], dtype=torch.float64).reshape(2, 1, 1)
h0 = torch.tensor([[0.4]], dtype=torch.float64)
outputs, final = gru(sequence, h0)

expected_h1 = 0.5 * 0.4 + 0.5 * math.tanh(0.8 + 0.5 * 0.4 * 0.3)
expected_h2 = 0.5 * expected_h1 + 0.5 * math.tanh(0.5 * expected_h1 * 0.3)
torch.testing.assert_close(outputs[0][0, 0], torch.tensor(expected_h1, dtype=torch.float64), atol=1e-12, rtol=0)
torch.testing.assert_close(final[0, 0], torch.tensor(expected_h2, dtype=torch.float64), atol=1e-12, rtol=0)
assert torch.equal(outputs[-1], final)
print(f"PASS — h1={expected_h1:.8f}, h2={expected_h2:.8f} (scalar analytical reference)")

PASS — h1=0.54812884, h2=0.31508170 (scalar analytical reference)


### 1.3 Reset-gate placement: a distinguishable analytical example

**Question.** Does the reset gate modify the previous state **before** recurrent matrix multiplication?

Choose one row state $H_0=[0.2,-0.4]$, reset gate $R=[0.25,0.75]$, and recurrent matrix

$$W_{hh}=\begin{bmatrix}0&1\\2&0\end{bmatrix}.$$

The intended recurrent candidate preactivation is

$$ (R\odot H_0)W_{hh}=[0.05,-0.30]W_{hh}=\boxed{[-0.60,0.05]}. $$

Resetting *after* multiplication would instead produce

$$R\odot(H_0W_{hh})=[0.25,0.75]\odot[-0.8,0.2]=[-0.2,0.15].$$

Clearly these differ. Setting all other weights and candidate bias to zero and $Z=0.5$, the independently predicted next state is $\frac12H_0+\frac12\tanh([-0.6,0.05])$.

In [13]:
# Check one-step hand-computable example distinguishes reset-before vs reset-after
gru = GRUScratch(1, 2, sigma=0).double()
with torch.no_grad():
    gru.b_r.copy_(torch.tensor([math.log(1/3), math.log(3)], dtype=torch.float64))
    gru.W_hh.copy_(torch.tensor([[0.0, 1.0], [2.0, 0.0]], dtype=torch.float64))

h0 = torch.tensor([[0.2, -0.4]], dtype=torch.float64)
inputs = torch.zeros((1, 1, 1), dtype=torch.float64)
_, observed, gates = gru(inputs, h0, record_gates=True)
expected = torch.tensor([
    0.5 * 0.2 + 0.5 * math.tanh(-0.6),
    0.5 * (-0.4) + 0.5 * math.tanh(0.05),
], dtype=torch.float64).reshape(1, 2)
torch.testing.assert_close(observed, expected, atol=1e-12, rtol=0)
torch.testing.assert_close(gates["r"][0, 0], torch.tensor([0.25, 0.75], dtype=torch.float64), atol=1e-12, rtol=0)
print("PASS — reset before matrix candidate and final state match analytical values")

PASS — reset before matrix candidate and final state match analytical values


### 1.4 Diagnostics, interventions and gradient propagation

**Computation.** `record_gates=True` returns detached $(T,B,K)$ arrays `z_computed`, `z_effective`, `r`. `z_override(X,H,Z,t)` can replace the **effective** update gate, leaving the computed gate available for comparison. With no intervention, $Z_t^{\mathrm{effective}}=Z_t^{\mathrm{computed}}$.

**Assumptions.** The override must return a tensor of shape $(B,K)$ and must not mutate its inputs. Shape is checked, but admissible numerical range, dtype and device are not fully validated; a replacement outside $[0,1]$ no longer has the usual convex-interpolation interpretation. Recorded diagnostics are detached; **the forward states and predictions are not detached** during ordinary differentiable execution.

**Analytical interventions.** If $Z_t^{\mathrm{effective}}\equiv1$, then $H_t=H_{t-1}$ for all $t$, regardless of the candidate. An identity override must agree with the ordinary run.

**Bounded-state invariant.** If $H_0\in[-1,1]^{B\times K}$ and gates have values in $[0,1]$, then $\widetilde H_t\in[-1,1]^{B\times K}$ (by tanh) and each $H_t$ is a coordinatewise convex combination, giving by induction the form $H_t\in[-1,1]^{B\times K}$.

In [14]:
# Check shape, override, detachment and empty-sequence tests.
gru = GRUScratch(1, 2, sigma=0).double()
inputs = torch.tensor([1.0, 0.0, 1.0]).reshape(3, 1, 1).double()
h0 = torch.tensor([[0.4, -0.2]], dtype=torch.float64)

normal_states, normal_final, normal_gates = gru(inputs, h0, record_gates=True)
identity_final = gru(inputs, h0, z_override=lambda X, H, Z, t: Z)[1]
assert torch.equal(normal_final, identity_final)
assert len(normal_states) == 3 and normal_final.shape == (1, 2)
assert normal_states[-1].requires_grad
for values in normal_gates.values():
    assert values.shape == (3, 1, 2) and not values.requires_grad
assert torch.equal(normal_gates["z_computed"], normal_gates["z_effective"])

_, preserved, forced_gates = gru(
    inputs, h0, record_gates=True,
    z_override=lambda X, H, Z, t: torch.ones_like(Z),)

assert torch.equal(preserved, h0)
assert torch.all(forced_gates["z_effective"] == 1)
empty_states, empty_final = gru(inputs[:0], h0)
assert empty_states == [] and torch.equal(empty_final, h0)
print("PASS — trajectories, detached gate records, identity/retention overrides, empty sequence")

PASS — trajectories, detached gate records, identity/retention overrides, empty sequence


In [15]:
# Central finite difference checks a nontrivial two-step parameter gradient
gru = GRUScratch(1, 1, sigma=0).double()
with torch.no_grad():
    gru.W_xh.fill_(0.8)
    gru.W_hh.fill_(0.3)
inputs = torch.tensor([1.0, 0.0], dtype=torch.float64).reshape(2, 1, 1)
h0 = torch.tensor([[0.4]], dtype=torch.float64)

gru.zero_grad()
gru(inputs, h0)[1][0, 0].backward()
autograd_grad = gru.W_xh.grad.item()

eps = 1e-6
original_weight = gru.W_xh.item()
with torch.no_grad():
    gru.W_xh.fill_(original_weight + eps)
    plus = gru(inputs, h0)[1][0, 0].item()
    gru.W_xh.fill_(original_weight - eps)
    minus = gru(inputs, h0)[1][0, 0].item()
    gru.W_xh.fill_(original_weight)
finite_difference = (plus - minus) / (2 * eps)
assert abs(autograd_grad - finite_difference) < 1e-7
print(f"PASS — parameter gradient: autograd={autograd_grad:.10f}, finite difference={finite_difference:.10f}")

PASS — parameter gradient: autograd=0.1479972367, finite difference=0.1479972367


## 2. CountingGRUSystem: recurrent representation and scalar readout

**Code correspondence (`counting_experiment.py`, lines 14–27):**

```python
self.gru = GRUScratch(1, num_hiddens, sigma)
self.readout = nn.Linear(num_hiddens, 1)
# In forward:
outputs, H_final = self.gru(X, H, record_gates, z_override)[:2]
prediction = self.readout(H_final).squeeze(-1)
```

The wrapper instantiates `GRUScratch(D=1, K, sigma)` and `nn.Linear(K,1)`. Starting from $H_0=0$ unless specified, it computes

$$H_t=F_\theta(X_t,H_{t-1}),\qquad \hat Y=H_TW_{\mathrm{out}}^\top+b_{\mathrm{out}}\in\mathbb R^{B\times1}.$$

A final `.squeeze(-1)` gives shape $(B,)$, retaining that dimension even for $B=1$. The readout weights have shape $(1,K)$ and bias $(1,)$. They use **PyTorch `nn.Linear` default initialisation**, not the GRU's `sigma` rule.

**Parameter count.** With $D=1,K=16$, the GRU has $864$ scalars and the affine readout has $16+1=17$, totaling **881 trainable scalars**.

**Why the prediction is continuous.** The supervised count is integer-valued, but the affine prediction is real-valued. Under eventual MSE training, a prediction $2.8$ for target $3$ incurs loss $(2.8-3)^2=0.04$. Rounding can be used to measure exact-count accuracy **at evaluation**, but inserting `round()` in ordinary backpropagation would give zero derivative almost everywhere. Integer targets do not require integer-valued internal states.

**Representation is not prescribed.** The wrapper does **not** implement $c_t=c_{t-1}+x_t$. If the model predicts counts accurately, the final hidden state must contain information accessible to its linear readout, but no single coordinate is required to equal the cumulative count. For instance, $h_{T,1}=0.1Y$ and readout weight $10$ could represent finite counts. This is a *possibility*, not a claim about what our trained GRU actually learns.

**Diagnostic mode.** `return_states=True` stacks detached $(T,B,K)$ states; `record_gates=True` adds detached gate trajectories. The returned prediction retains autograd connectivity. `_model_input(model,X)` aligns tensor dtype/device to the first model parameter; this is execution plumbing, not a change to the statistical task.

**Analytical example.** With a zero-parameter GRU and $H_0=[0.4,-0.2]$, $Z=R=0.5$, $\widetilde H_t=0$, hence $H_2=[0.1,-0.05]$. Readout $W_{\mathrm{out}}=[2,-3]$, $b_{\mathrm{out}}=0.5$ gives $\hat y=0.1(2)+(-0.05)(-3)+0.5=0.85$.

In [16]:
# Controlled wrapper test: manual hidden-state decay and affine readout
model = CountingGRUSystem(num_hiddens=2, sigma=0).double()
with torch.no_grad():
    for parameter in model.gru.parameters():
        parameter.zero_()
    model.readout.weight.copy_(torch.tensor([[2.0, -3.0]], dtype=torch.float64))
    model.readout.bias.fill_(0.5)

inputs = torch.tensor([1.0, 0.0]).reshape(2, 1, 1).double()
h0 = torch.tensor([[0.4, -0.2]], dtype=torch.float64)
prediction, info = model(inputs, H=h0, return_states=True, record_gates=True)
expected_states = torch.tensor([[[0.2, -0.1]], [[0.1, -0.05]]], dtype=torch.float64)
torch.testing.assert_close(info["states"], expected_states, atol=1e-12, rtol=0)
torch.testing.assert_close(prediction, torch.tensor([0.85], dtype=torch.float64), atol=1e-12, rtol=0)
assert prediction.shape == (1,) and prediction.requires_grad
assert not info["states"].requires_grad and not info["H_final"].requires_grad
assert all(not gate.requires_grad for gate in info["gates"].values())

model.zero_grad()
prediction.sum().backward()
torch.testing.assert_close(model.readout.weight.grad, expected_states[-1], atol=1e-12, rtol=0)
torch.testing.assert_close(model.readout.bias.grad, torch.ones(1, dtype=torch.float64))

standard_model = CountingGRUSystem(16)
assert sum(p.numel() for p in standard_model.parameters()) == 881
assert standard_model(torch.zeros(2, 3, 1)).shape == (3,)
assert _model_input(model, torch.ones(1, dtype=torch.float32)).dtype == torch.float64
print("PASS — counting wrapper, readout=0.85, gradient and detached diagnostics, 881 parameters")

PASS — counting wrapper, readout=0.85, gradient and detached diagnostics, 881 parameters


## 3. `generate_counting_batch`: defining the statistical problem

### 3.1 Mathematical sampling model and target definition

For a fixed length $T$, batch size $B$, and token probability $p\in[0,1]$, define for $t=1,\ldots,T$ and $b=1,\ldots,B$:

$$x_{t,b}\overset{\mathrm{iid}}\sim\operatorname{Bernoulli}(p),\qquad
\Pr(x_{t,b}=1)=p,\quad \Pr(x_{t,b}=0)=1-p.$$

Prefix counts and terminal targets are **deterministic** given the sampled tokens:

$$C_{t,b}=\sum_{s=1}^tX_{s,b},\qquad Y_b=C_{T,b}=\sum_{t=1}^{T}x_{t,b}.$$

**Analytical example:** tokens $[0,1,0,1,1]$ produce count prefixes $[0,1,1,2,3]$ and final target $3$.

**Shapes:** `X: (T,B,1)` (float32 binary values), `y_steps: (T,B)` and `target: (B,)`. The function returns `(X, y_steps[-1], y_steps)`.

### 3.2 Sampling construction, line by line

**Code correspondence (`counting_experiment.py`, lines 56–63):**

```python
X = (torch.rand(T, batch_size, 1, generator=generator) < p).to(
    device=device, dtype=torch.float32
)
y_steps = X[..., 0].cumsum(0)
return X, y_steps[-1], y_steps
```

1. Require $T\geq1$, $B\geq1$ and $0\leq p\leq1$; the code assumes integer dimensions.
2. Draw $U_{t,b}\sim\operatorname{Uniform}[0,1)$ with `torch.rand(T,B,1, generator=generator)`.
3. Threshold: $x_{t,b}=\mathbf1[U_{t,b}<p]$. Under ideal uniform draws, $\Pr(x_{t,b}=1)=p$.
4. Convert the Boolean result to float32 and move it to the requested device. The optional generator is a **CPU generator**; sampling occurs on CPU before transfer.
5. Compute `X[..., 0].cumsum(0)`: remove the singleton feature dimension and sum **over time**, not across examples.
6. Return the final row as the terminal count. Integer values up to our tested horizons are exactly representable in float32.

The model later receives `X`, not the target. The prefix counts are provided for diagnostics; the current counting training objective uses only the terminal target.

### 3.3 Distribution of the target (derivation)

Since $Y_b$ is a sum of $T$ iid Bernoulli variables, **conditional on length**,

$$Y\mid T\sim\operatorname{Binomial}(T,p),$$

$$\Pr(Y=k\mid T)=\binom Tk p^k(1-p)^{T-k},\qquad k=0,\ldots,T.$$

Linearity of expectation gives

$$\mathbb E[Y\mid T]=\sum_{t=1}^T\mathbb E[X_t]=\boxed{pT}.$$

Independence gives

$$\operatorname{Var}(Y\mid T)=\sum_{t=1}^T\operatorname{Var}(X_t)=\boxed{Tp(1-p)}.$$

**For $p=0.5$ and $T=10$:** the mean count is $5$ and the variance is $2.5$. Every observed sequence has an **exact** integer target; $pT$ is only the conditional *mean*, not the correct count for an arbitrary sample.

In [17]:
# Boundary cases plus independent Python-integer reconstruction of targets.
for p in (0.0, 1.0):
    X, terminal, prefixes = generate_counting_batch(4, batch_size=3, p=p)
    assert X.shape == (4, 3, 1)
    assert torch.all(X == p)
    assert torch.all(terminal == 4 * p)
    for t in range(4):
        assert torch.all(prefixes[t] == (t + 1) * p)

X, terminal, prefixes = generate_counting_batch(
    7, batch_size=4, p=0.5,
    generator=torch.Generator().manual_seed(42),
)
assert X.shape == (7, 4, 1)
assert terminal.shape == (4,) and prefixes.shape == (7, 4)
assert X.dtype == torch.float32
for b in range(4):
    running = 0
    for t in range(7):
        token = int(X[t, b, 0].item())
        assert token in (0, 1)
        running += token
        assert prefixes[t, b].item() == running
    assert terminal[b].item() == running
print("PASS — exact boundary cases and Python-integer prefix/final target reconstruction")

PASS — exact boundary cases and Python-integer prefix/final target reconstruction


In [18]:
# Statistical sanity check
T, B, p = 10, 20_000, 0.5
_, targets, _ = generate_counting_batch(
    T, batch_size=B, p=p,
    generator=torch.Generator().manual_seed(2026),
)
theoretical_mean = T * p
theoretical_variance = T * p * (1 - p)
observed_mean = targets.mean().item()
observed_variance = targets.var(unbiased=False).item()
mean_standard_error = math.sqrt(theoretical_variance / B)
assert abs(observed_mean - theoretical_mean) < 5 * mean_standard_error
assert abs(observed_variance - theoretical_variance) < 0.15
print(f"PASS — Binomial(10, 0.5) sample: mean={observed_mean:.4f} (theory 5.0000), "
      f"variance={observed_variance:.4f} (theory 2.5000)")

PASS — Binomial(10, 0.5) sample: mean=5.0082 (theory 5.0000), variance=2.4788 (theory 2.5000)


### 3.4 A length-only predictor is statistically legitimate—but not exact counting

The prediction $\hat Y=pT$ uses length, not the realised token values. Under squared loss, it is the optimal predictor among functions that observe **only $T$**:

$$\mathbb E[(Y-pT)^2\mid T]=\operatorname{Var}(Y\mid T)=Tp(1-p).$$

For example, $p=0.5$, $T=10$ gives mean-squared error $2.5$ *in expectation*, while the exact oracle $\sum_tX_t$ has error zero for every sequence.

**Conditional illustration for the later training review.** *If* lengths are uniformly sampled from $\{5,\ldots,20\}$ and $p=0.5$, then

$$\mathbb E[T]=12.5,\quad\operatorname{Var}(T)=21.25,\quad
\mathbb E[Y]=p\mathbb E[T]=6.25.$$

The expected MSE of the length-only predictor is

$$\mathbb E[Tp(1-p)]=3.125.$$

The expected MSE of the best **constant** predictor $\hat Y=6.25$ follows the law of total variance:

$$\operatorname{Var}(Y)=\mathbb E[\operatorname{Var}(Y\mid T)]+\operatorname{Var}(\mathbb E[Y\mid T])=3.125+0.25(21.25)=8.4375.$$

These are theoretical baselines **under the stated length distribution**, not observed model results and not a training-protocol verification. The actual distribution of $T$ during optimisation will be reviewed when we reach `train_counting()`.

In [19]:
# Pure arithmetic check of the conditional illustrative baseline values
lengths = list(range(5, 21))
mean_T = sum(lengths) / len(lengths)
var_T = sum((T - mean_T) ** 2 for T in lengths) / len(lengths)
length_only_mse = sum(T / 4 for T in lengths) / len(lengths)
constant_mse = length_only_mse + var_T / 4
assert (mean_T, var_T, length_only_mse, constant_mse) == (12.5, 21.25, 3.125, 8.4375)
print("PASS — conditional theory: E[T]=12.5, Var(T)=21.25, length-only MSE=3.125, constant MSE=8.4375")

PASS — conditional theory: E[T]=12.5, Var(T)=21.25, length-only MSE=3.125, constant MSE=8.4375


**Notes for §3.** The generator implements Bernoulli threshold sampling and **exact target construction**. The task is statistically well defined for a fixed $T,B,p$. The count–length correlation $\mathbb E[Y\mid T]=pT$ is a property of the distribution; it is not label leakage or a model implementation defect. A statistical sample check supports the intended distribution but cannot prove independence or randomness.

## 4. `length_batches`: fixed-length evaluation collections

Given an **ordered** collection of distinct lengths $\mathcal T=(T_1,\ldots,T_m)$, it constructs

$$\mathcal B=\{T\mapsto X^{(T)}:T\in\mathcal T\},\qquad
X^{(T)}\in\{0,1\}^{T\times B\times1},\qquad
x^{(T)}_{t,b}\sim\operatorname{Bernoulli}(p).$$

**Code correspondence (`counting_experiment.py`, lines 66–70):**

```python
generator = torch.Generator().manual_seed(seed)
return {
    T: generate_counting_batch(T, batch_size, p, device, generator)[0]
    for T in lengths
}
```

**Implementation:** `torch.Generator().manual_seed(seed)` creates one local CPU RNG; a dictionary comprehension calls `generate_counting_batch(...)[0]` **once per listed length**. The `[0]` keeps only the input tensor; later evaluation functions can recover its true counts as $Y_b^{(T)}=\sum_tx^{(T)}_{t,b}$.

**Example.** With lengths `(5, 10, 20)` and batch size 256, the values have shapes `(5,256,1)`, `(10,256,1)`, `(20,256,1)`.

**Reproducibility assumptions and caveats.** Identical seed, arguments, and **order of lengths** reproduce the same tensors in a fixed runtime. One generator is advanced across lengths, so changing the order can change the samples associated with a given $T$. Duplicate keys overwrite earlier entries in the dictionary; use distinct lengths. This function is **not** a training curriculum and does **not** choose the training distribution over lengths.

**Scientific implication.** At fixed $p$, increasing $T$ changes both the number of recurrent transitions and the count distribution:

$$\mathbb E[Y\mid T]=pT,\qquad \operatorname{Var}(Y\mid T)=Tp(1-p).$$

Therefore, ordinary long-length evaluations confound longer computational horizon with larger expected count. This is not a sampling bug. Later **exact-count controls** may help separate aspects of these effects.

In [20]:
# Shapes, deterministic endpoints, local-seed reproducibility, and ordering caveat.
requested = (5, 10, 20)
for p in (0.0, 1.0):
    batches = length_batches(requested, batch_size=8, p=p, seed=91)
    assert tuple(batches) == requested
    for T, X in batches.items():
        assert X.shape == (T, 8, 1)
        assert torch.all(X == p)

first = length_batches(requested, batch_size=8, p=0.5, seed=91)
repeat = length_batches(requested, batch_size=8, p=0.5, seed=91)
reordered = length_batches(tuple(reversed(requested)), batch_size=8, p=0.5, seed=91)
assert all(torch.equal(first[T], repeat[T]) for T in requested)
assert all(torch.all((first[T] == 0) | (first[T] == 1)) for T in requested)
assert not torch.equal(first[5], reordered[5])  # Same seed; different RNG consumption order.
print("PASS — per-length shapes, boundary tokens, deterministic replay, order-sensitive sampling")

PASS — per-length shapes, boundary tokens, deterministic replay, order-sensitive sampling


## 5. `exact_count_batch`: exact count recurrence generation

**Note** This function generates binary sequences with a fixed, precisely controlled number of ones. This is an important methodological component of the experiment because, unlike ordinary Bernoulli sampling, it allows us to specify both sequence length and target count.

**Maths.** The function constructs $B$ binary sequences, each satisfying two constraints
$$
X^{(b)}\in\{0,1\}^{T}
$$

and
$$
\sum_{t=1}^{T}X_{t,b}=k,
$$

where $k=\texttt{count}$. For example, suppose $ T=8, k=3$, then every generated sequence must contain exactly three ones and five zeros.

The function returns only the input tensor X, with shape $(T,B,1)$.

**Definition.**
Let
$$
\Omega_{T,k}
=
\left\{
x\in\{0,1\}^{T}:
\sum_{t=1}^{T}x_t=k
\right\}.
$$

This is the set of all binary sequences of length $T$ containing exactly $k$ ones.

The number of distinct sequences is

$$
|\Omega_{T,k}|=\binom{T}{k}.
$$

Why? Because a sequence is fully determined by choosing which $k$ of the $T$ positions contain ones.

**Random ordering.**

Under ideal continuous random sampling, the random condition selects every valid arrangement with equal probability:
$$
P(X=x\mid T,k)
=
\frac{1}{\binom Tk},
\quad x\in\Omega_{T,k}.
$$

This is a uniform distribution over all sequences with the specified length and count. There is an important relationship to our earlier Bernoulli task. Suppose
$$
X_t\overset{\mathrm{iid}}{\sim}\operatorname{Bernoulli}(p)
$$

for $0<p<1$. For any particular sequence x containing k ones,
$$
P(X=x)=p^k(1-p)^{T-k}.
$$

The probability that the total count equals $k$ is
$$
P(Y_T=k)=\binom Tk p^k(1-p)^{T-k}.
$$

Therefore,
$$
\begin{aligned}
P(X=x\mid Y_T=k)
&=\frac{P(X=x)}{P(Y_T=k)}\\
&=\frac{p^k(1-p)^{T-k}}
{\binom Tk p^k(1-p)^{T-k}}\\
&=\frac1{\binom Tk}.
\end{aligned}
$$

This is a central statistical finding. The random exact-count distribution is the same as the original iid Bernoulli distribution conditioned on a particular final count. This holds for any $0<p<1$.

However, once we condition on the count, the token positions are no longer statistically independent. For example, if T=4 and exactly three ones must appear, seeing three ones in the first three positions guarantees the final token is zero.

In [21]:
from counting_experiment import exact_count_batch

expected_positions = {
    "front": [0, 1, 2],
    "back": [7, 8, 9],
    "spread": [0, 4, 9],
}

for order, expected in expected_positions.items():
    X = exact_count_batch(T=10, count=3, batch_size=4, order=order)

    assert X.shape == (10, 4, 1)

    for b in range(4):
        positions = torch.nonzero(X[:, b, 0]).flatten().tolist()
        assert positions == expected

# Fixed-count invariant under random positioning
X = exact_count_batch(
    T=20, count=6, batch_size=128, order="random",
    generator=torch.Generator().manual_seed(42),
)

assert all(
    sum(int(X[t, b, 0]) for t in range(20)) == 6
    for b in range(128)
)

print("Exact-count generator checks passed.")

Exact-count generator checks passed.


**Notes for §5.** This is an important part of the methodology hen it comes to the experimental design. By fixing the count, we can compare by holding the true target count constant. Changing length while holding count fixed also changes the frequency and spacing of zero tokens, as well as the rarity of the event under the original Bernoulli training distribution. Therefore, if the GRU gives different predictions to these different lengths, we can establish that the model's prediction is not invariant to these length changes when the true count is fixed.

## 5. `controlled_batches`
This function assembles exact-count sequences into four families of controlled evaluation conditions.

**Source:** `counting_experiment.py`, lines 101–120.

The function constructs **17 evaluation conditions** using `exact_count_batch()`. These are behavioural diagnostics, not training stages.

### i. Mathematical definitions

Let a binary sequence be

$$
x=(x_1,\ldots,x_T)\in\{0,1\}^T,
$$

with sequence length $T$, exact count $k$, and empirical one-frequency $f$:

$$
k=\sum_{t=1}^{T}x_t,
\qquad
f=\frac{k}{T}.
$$

The set of all sequences with fixed length $T$ and count $k$ is

$$
\Omega_{T,k}
=
\left\{
x\in\{0,1\}^{T}:
\sum_{t=1}^{T}x_t=k
\right\},
$$

with cardinality

$$
|\Omega_{T,k}|=\binom{T}{k}.
$$

For `order="random"`, the intended sampling distribution is

$$
X\sim\operatorname{Uniform}(\Omega_{T,k}).
$$

This is equivalent to iid Bernoulli sampling conditioned on the final count, for $0<p<1$:

$$
P(X=x\mid Y_T=k,T)
=
\frac{1}{\binom{T}{k}},
\qquad x\in\Omega_{T,k}.
$$

Conditional on the count, token positions are no longer independent.

Importantly, **empirical frequency $f=k/T$ is not the Bernoulli**.

### ii. Controlled evaluation families

| Family | Held fixed | Varied | Conditions |
|---|---|---|---|
| Matched count | $k=6$ | $T$ | $T\in\{10,20,40,80\}$ |
| Matched length | $T=20$ | $k$ | $k\in\{0,2,6,10,20\}$ |
| Matched frequency | $f=0.25$ | $T$ and $k$ | $T\in\{8,20,40,80\}$, $k=T/4$ |
| Matched multiset | $T=40$, $k=6$ | Token arrangement | Front, back, spread, random |

All conditions use the same binary alphabet, $\{0,1\}$.

#### A. Matched count

$$
X^{(T)}\sim\operatorname{Uniform}(\Omega_{T,6}),
\qquad
T\in\{10,20,40,80\}.
$$

Every sequence has the same target:

$$
Y=6.
$$

**Question:** Does the model maintain the correct count prediction as the sequence becomes longer?

Holding $k$ fixed does not isolate length alone. The number of zeros, empirical frequency, and distribution of gaps between ones also change:

| $T$ | $k$ | Zeros | $f=k/T$ |
|---|---|---|---|
| 10 | 6 | 4 | 0.60 |
| 20 | 6 | 14 | 0.30 |
| 40 | 6 | 34 | 0.15 |
| 80 | 6 | 74 | 0.075 |

A prediction difference establishes sensitivity to these changed sequence conditions, but does not by itself identify the responsible recurrent mechanism.

#### B. Matched length

$$
X^{(k)}\sim\operatorname{Uniform}(\Omega_{20,k}),
\qquad
k\in\{0,2,6,10,20\}.
$$

The processing horizon is fixed at $T=20$, while the target changes:

$$
Y=k.
$$

**Question:** Can the GRU accurately distinguish different count magnitudes at a fixed sequence length?

Because

$$
f=\frac{k}{20},
$$

changing $k$ also changes empirical frequency.

The extreme cases $k=0$ and $k=20$ are within the support of the Bernoulli(0.5) training distribution, but are exceptionally rare:

$$
P(Y=0\mid T=20)
=
P(Y=20\mid T=20)
=
2^{-20}.
$$

Thus, a length within the training range does not necessarily imply a typical token composition.

#### C. Matched frequency

$$
X^{(T)}
\sim\operatorname{Uniform}(\Omega_{T,T/4}),
\qquad
T\in\{8,20,40,80\}.
$$

The empirical one-frequency is fixed:

$$
f=\frac{k}{T}=0.25.
$$

| $T$ | $k$ | $f$ |
|---|---|---|
| 8 | 2 | 0.25 |
| 20 | 5 | 0.25 |
| 40 | 10 | 0.25 |
| 80 | 20 | 0.25 |

**Question:** How does the model behave when sequence length and required count increase together at constant empirical frequency?

These sequences are **not iid Bernoulli(0.25)** samples. They contain exactly $T/4$ ones, so their token positions are dependent.

This condition also differs from the expected empirical frequency of the training distribution, where $p=0.5$.

#### D. Matched multiset

All conditions satisfy

$$
T=40,\qquad k=6,\qquad f=0.15.
$$

The same multiset of six ones and 34 zeros is arranged in four ways:

- **Front:** ones occupy the first six positions.
- **Back:** ones occupy the last six positions.
- **Spread:** ones are distributed approximately evenly.
- **Random:** six distinct positions are sampled uniformly under the intended sampling model.

For `spread`, the zero-based positions are

$$
(0,8,16,23,31,39).
$$

**Question:** Is the learned prediction invariant to rearranging tokens when the correct count is unchanged?

Exact counting is permutation-invariant:

$$
\sum_{t=1}^{T}x_t
=
\sum_{t=1}^{T}x_{\pi(t)}
$$

for any permutation $\pi$.

An exact counting predictor therefore satisfies

$$
g(x_1,\ldots,x_T)
=
g(x_{\pi(1)},\ldots,x_{\pi(T)}).
$$

Prediction differences across arrangements establish a failure of permutation invariance, but do not identify its internal cause.

### iii. Mathematical properties of an exact counter

The controlled conditions assess several necessary behavioural properties of exact counting.

**Count accuracy**

$$
g(x_{1:T})=\sum_{t=1}^{T}x_t.
$$

**Zero-extension invariance**

For every $m\geq0$,

$$
g(x_1,\ldots,x_T,\underbrace{0,\ldots,0}_{m})
=
g(x_1,\ldots,x_T).
$$

**Permutation invariance**

$$
g(x_{1:T})=g(x_{\pi(1):\pi(T)}).
$$

These are properties of an exact counting function, not properties guaranteed by a trained GRU.

The matched count family is relevant to count preservation at different lengths, but is not a paired zero-extension experiment. A direct zero-extension test requires preserving the original prefix and appending zeros to that same sequence.

Likewise, the matched multiset family tests selected arrangements; it does not exhaustively verify permutation invariance over all possible arrangements.

### iv. Important assumptions and methodological findings

**Relationship between length, count and frequency**

The quantities satisfy

$$
\boxed{k=fT}.
$$

They cannot all vary independently. Holding any two fixed determines the third. Our four families are complementary comparisons rather than independently manipulated causal factors.

**Overlapping conditions**

Some differently labelled batches share the same mathematical distribution. For example,

- `matched_count/T=20/k=6`
- `matched_length/T=20/k=6`

are separately sampled from $\operatorname{Uniform}(\Omega_{20,6})$.

Likewise, `matched_count/T=40/k=6` and `matched_multiset/order=random` share the same distribution.

These labels describe different planned comparisons, not necessarily different underlying data populations.

**Reproducibility**

The function uses a locally seeded CPU random-number generator, shared across specifications. Results are reproducible for fixed seeds, ordered specifications, and a compatible execution environment.

# Training

## 1. Training the counting GRU — `train_counting()`

**Source:** `counting_experiment.py`, lines 123–191.

The purpose of this section is to verify the mathematical and computational correctness of the training procedure.

**The training objective is final-count prediction.** It does not explicitly supervise intermediate hidden states or require the GRU to implement an exact accumulation algorithm.

### 1.1. Motivating questions

1. What population objective is the counting GRU trained to minimise?
2. How are sequence lengths and binary tokens sampled during training?
3. How does the finite-minibatch loss approximate the population objective?
4. How are the GRU and readout parameters updated?
5. How are training and validation data separated?
6. What do the reported training and validation losses actually measure?
7. What can we conclude from successful optimisation, and what remains unestablished?

### 1.2. Mathematical definitions and notation

Let

- $T$ denote sequence length.
- $B$ denote minibatch size.
- $K=16$ denote hidden-state dimension.
- $X_{t,b}\in\{0,1\}$ denote the token at timestep $t$ for example $b$.
- $Y_T^{(b)}$ denote the final true count.
- $f_\theta$ denote the complete GRU and linear-readout model.
- $\theta$ denote all trainable GRU and readout parameters.
- $\hat Y_T^{(b)}=f_\theta(X_{1:T}^{(b)})$ denote the predicted count.
- $u$ denote the optimiser-update index.

The final-count target is

$$
\boxed{
Y_T^{(b)}=\sum_{t=1}^{T}X_{t,b}.
}
$$

The model computes

$$
H_t=F_{\theta_{\mathrm{GRU}}}(X_t,H_{t-1}),
$$

followed by

$$
\boxed{
\hat Y_T=H_TW_{\mathrm{out}}^\top+b_{\mathrm{out}}.
}
$$

Here the readout expression uses batched, row-oriented tensor notation.

The model produces continuous predictions:

$$
\hat Y_T^{(b)}\in\mathbb R.
$$

Integer rounding is **not** performed during training.

---

### 1.3. Training-data distribution

At each optimiser update, the sequence length is sampled uniformly from

$$
\mathcal T_{\mathrm{ID}}=\{5,6,\ldots,20\}.
$$

Thus,

$$
\boxed{
P(T=t)=\frac{1}{16},
\qquad t\in\mathcal T_{\mathrm{ID}}.
}
$$

Conditional on the sampled sequence length, the input tokens are generated according to

$$
\boxed{
X_{t,b}\overset{\mathrm{iid}}{\sim}
\operatorname{Bernoulli}(p),
\qquad p=0.5.
}
$$

The conditional probability of a particular sequence is

$$
P(X_{1:T}=x_{1:T}\mid T)
=
\prod_{t=1}^{T}
p^{x_t}(1-p)^{1-x_t}.
$$

The final count consequently satisfies

$$
Y_T\mid T\sim\operatorname{Binomial}(T,p).
$$

Its conditional expectation and variance are

$$
\boxed{
\mathbb E[Y_T\mid T]=pT
}
$$

and

$$
\boxed{
\operatorname{Var}(Y_T\mid T)=Tp(1-p).
}
$$

**Implementation detail:** One sequence length is sampled per optimiser update. All $B$ examples in that minibatch share the same length, although their token sequences are sampled separately.

The training procedure is therefore *length-homogeneous within each minibatch*, but variable-length across optimiser updates.

Note: the length distribution is unchanged throughout training, and so this is not a curriculum. To keep this experiment and its notes clean, curriculum-style token counting set-ups are considered in a separate experiment.

---

### 1.4. Population training objective

The model is trained to minimise squared error between its final count prediction and the true count.

The population objective is

$$
\boxed{
\mathcal R(\theta)
=
\mathbb E_{T}
\left[
\mathbb E_{X_{1:T}\mid T}
\left[
\left(f_\theta(X_{1:T})-Y_T\right)^2
\right]
\right].
}
$$

Since sequence lengths are uniformly sampled,

$$
\boxed{
\mathcal R(\theta)
=
\frac{1}{16}
\sum_{T=5}^{20}
\mathbb E_{X_{1:T}\mid T}
\left[
\left(
f_\theta(X_{1:T})
-
\sum_{t=1}^{T}X_t
\right)^2
\right].
}
$$

This expression defines the statistical learning objective of our initial counting experiment.

**Important observation:** Every sequence length contributes equally to the population objective. The objective is not weighted by sequence length, token count, or the number of possible binary sequences at a particular length.

---

### 1.5. Finite-minibatch loss

At optimiser update $u$, a length $T_u$ is sampled, followed by $B$ binary sequences of that length.

The minibatch objective is

$$
\boxed{
\widehat{\mathcal L}_u(\theta)
=
\frac{1}{B}
\sum_{b=1}^{B}
\left(
f_\theta(X_{1:T_u}^{(b)})
-
Y_{T_u}^{(b)}
\right)^2.
}
$$

The corresponding implementation is

```python
loss = (model(X) - target).square().mean()
```

This correctly computes mean squared error over the minibatch.

Under the intended sampling distribution,

$$
\mathbb E[\widehat{\mathcal L}_u(\theta)]
=
\mathcal R(\theta)
$$

for fixed model parameters $\theta$.

Subject to the usual regularity conditions for exchanging differentiation and expectation, the minibatch gradient is also an unbiased estimator of the population-risk gradient

$$
\mathbb E[
\nabla_\theta\widehat{\mathcal L}_u(\theta)
]
=
\nabla_\theta\mathcal R(\theta).
$$

#### Analytical example

Suppose the true counts and predictions for a minibatch of two examples are

$$
Y=(2,2),
\qquad
\hat Y=(1.5,2.3).
$$

Then

$$
\begin{aligned}
\widehat{\mathcal L}
&=
\frac{(1.5-2)^2+(2.3-2)^2}{2}\\
&=
\frac{0.25+0.09}{2}\\
&=\boxed{0.17}.
\end{aligned}
$$

The gradient with respect to the predicted count for example $b$ is

$$
\boxed{
\frac{\partial\widehat{\mathcal L}}{\partial\hat Y_b}
=
\frac{2}{B}(\hat Y_b-Y_b).
}
$$

For this example,

$$
\nabla_{\hat Y}\widehat{\mathcal L}
=
(-0.5,0.3).
$$

The loss encourages the first prediction to increase and the second to decrease.

---

### 1.6. Parameter initialisation and optimisation

The training function constructs

```python
model = CountingGRUSystem(
    num_hiddens=16,
    sigma=0.01
).to(device)
```

The complete model contains 881 trainable scalar parameters, as previously discussed, with

$$
N_{\mathrm{readout}}=K+1=17.
$$

The GRU uses the previously verified Gaussian weight initialisation with $\sigma=0.01$ and zero biases. The readout uses PyTorch's default `nn.Linear` initialisation. The optimiser is

```python
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)
```

All trainable parameters of the GRU and readout are included in the optimisation.

At each update, the implementation performs

```python
optimizer.zero_grad(set_to_none=True)
loss.backward()
optimizer.step()
```

As we emply an Adam-trained GRU, these operations:

1. Clear gradients from the previous update.
2. Compute gradients by backpropagation through the recurrent network.
3. Apply Adam's parameter update.

Adam maintains first- and second-moment estimates of the gradient:

$$
m_u=\beta_1m_{u-1}+(1-\beta_1)g_u,
$$

$$
v_u=\beta_2v_{u-1}+(1-\beta_2)g_u^2,
$$

where

$$
g_u=\nabla_\theta\widehat{\mathcal L}_u(\theta_{u-1}).
$$

With bias-corrected estimates $\hat m_u$ and $\hat v_u$, the update is

$$
\boxed{
\theta_u
=
\theta_{u-1}
-
\eta\frac{\hat m_u}{\sqrt{\hat v_u}+\epsilon}.
}
$$

All operations on the vectors above are coordinatewise where appropriate. The implementation uses PyTorch's Adam defaults apart from the specified learning rate. To keep this particular experiment vanilla, there is no learning-rate schedule, weight decay, gradient clipping, checkpoint selection, or early stopping in the current training procedure.

### 1.7. Random seeds and reproducibility

Three seeds have distinct roles:

| Seed | Default | Purpose |
|---|---|---|
| `seed` | 42 | Model parameter initialisation |
| `data_seed` | 43 | Training-length and token sampling |
| `val_seed` | 44 | Validation token sampling |

Model initialisation is preceded by

```python
torch.manual_seed(seed)
```

This sets PyTorch's global random-number-generator state.

The training and validation sampling streams are constructed separately:

```python
train_rng = torch.Generator().manual_seed(data_seed)
val_rng = torch.Generator().manual_seed(val_seed)
```

These are distinct pseudorandom streams.

**Important distinction:** Independently sampled training and validation sequences need not be unique or disjoint.

For example, at $T=5$, only $2^5=32$ distinct binary sequences exist. Repeated sequences and overlap between training and validation are therefore entirely possible.

This does not constitute an implementation error: training and validation are sampling from the same intended data-generating distribution.

For strict computational reproducibility, the software environment, device and relevant numerical settings must also be considered.

---

### 1.8. Fixed validation dataset

Before optimisation, the code generates 256 validation sequences for every length $T\in\{5,\ldots,20\}.$ Consequently,

$$
N_{\mathrm{val}}=16\times256=4096.
$$

The validation set is generated once and reused during training. The empirical validation objective at update $u$ is

$$
\boxed{
\widehat{\mathcal R}_{\mathrm{val}}(\theta_u)
=
\frac{1}{4096}
\sum_{T=5}^{20}
\sum_{b=1}^{256}
\left(
f_{\theta_u}(X_{1:T}^{(b)})
-
Y_T^{(b)}
\right)^2.
}
$$

The validation objective weights each length equally, consistent with the training-length distribution.

Because the examples remain fixed, evaluations at different updates are directly comparable on the same dataset.

Validation is performed using

```python
model.eval()

with torch.no_grad():
    errors = torch.cat([
        model(Xv) - yv
        for Xv, yv in validation
    ])
    val_mse = errors.square().mean().item()
```

This correctly computes pooled validation MSE.

However, validation monitoring does not itself establish generalisation, and this function does not construct a separate held-out final test set.

**ID versus OOD**

Given the training length range $T$, by design some controlled sequences fall within that range; others exceed it. Even within the training-length range, exact-count controls can select rare token compositions.

We must distinguish length extrapolation from atypical token composition and larger target magnitudes.


---

### 1.9. Interpretation of logged metrics

The history contains

```python
{
    "update": step,
    "train_mse": ...,
    "val_mse": ...
}
```

These two quantities have different interpretations.

#### Mean online training loss

The reported `train_mse` averages minibatch losses since the previous logging event.

For an interval containing $m$ updates ending at $u$,

$$
\boxed{
\overline{\mathcal L}_{\mathrm{train}}
=
\frac1m
\sum_{j=u-m+1}^{u}
\widehat{\mathcal L}_j(\theta_{j-1}).
}
$$

The losses were calculated at different parameter values and on freshly sampled minibatches.

Therefore, this is a **mean online training loss**, not the MSE of the current model on a fixed training dataset.

#### Fixed set validation loss

The reported `val_mse` is the empirical MSE of the model after update $u$, evaluated on the same 4,096 validation sequences.

Thus, the two metrics cannot be interpreted as a conventional fixed-dataset training-versus-validation comparison without this qualification.

The final logging interval may contain fewer than `log_every` updates; the implementation correctly divides by the actual number of updates in that interval.

---

### 1.10. Analytical population benchmarks

Before interpreting learned model performance, we can derive reference predictors from the specified training distribution.

These benchmarks are **mathematical reference values**, not predictors implemented by `train_counting()`.

For

$$
T\sim\operatorname{Uniform}\{5,\ldots,20\}
$$

and

$$
X_t\sim\operatorname{Bernoulli}(0.5),
$$

we have

$$
\mathbb E[T]=12.5
$$

and

$$
\operatorname{Var}(T)=\frac{16^2-1}{12}=21.25.
$$

#### Constant predictor

The optimal constant predictor under MSE is

$$
\boxed{
\hat Y_{\mathrm{constant}}
=
\mathbb E[Y_T]
=
p\mathbb E[T]
=
6.25.
}
$$

By the law of total variance,

$$
\operatorname{Var}(Y_T)
=
\mathbb E[\operatorname{Var}(Y_T\mid T)]
+
\operatorname{Var}(\mathbb E[Y_T\mid T]).
$$

Since

$$
\operatorname{Var}(Y_T\mid T)=Tp(1-p),
$$

we obtain

$$
\begin{aligned}
\mathcal R_{\mathrm{constant}}
&=p(1-p)\mathbb E[T]
+p^2\operatorname{Var}(T)\\
&=3.125+5.3125\\
&=\boxed{8.4375}.
\end{aligned}
$$

#### Length-only predictor

The optimal MSE predictor when observing only $T$ is

$$
\boxed{
\hat Y_{\mathrm{length}}=pT=T/2.
}
$$

Its population MSE is

$$
\begin{aligned}
\mathcal R_{\mathrm{length}}
&=\mathbb E[\operatorname{Var}(Y_T\mid T)]\\
&=p(1-p)\mathbb E[T]\\
&=\boxed{3.125}.
\end{aligned}
$$

#### Exact counting oracle

The exact counting algorithm predicts

$$
\boxed{
\hat Y_{\mathrm{oracle}}
=
\sum_{t=1}^{T}X_t.
}
$$

It achieves

$$
\boxed{
\mathcal R_{\mathrm{oracle}}=0.
}
$$

| Reference predictor | Population MSE |
|---|---:|
| Constant mean count, $6.25$ | 8.4375 |
| Length-only, $T/2$ | 3.125 |
| Exact counting oracle | 0 |

These are population expectations. Empirical values on finite validation or test samples will fluctuate around them.

The length-only predictor is a useful statistical benchmark but is not itself evidence of an exact counting procedure.

In [ ]:
"""Training verification: 
performs just one optimiser update and uses independently reconstructed targets and reference metrics. 
This is run a full training study or attempt to establish ID learning.
"""

import math
import torch

from counting_experiment import (
    CountingGRUSystem,
    generate_counting_batch,
    train_counting,
)

# Fixed CPU configuration for reproducibility.
seed, data_seed, val_seed = 71, 72, 73
batch_size = 8

with torch.random.fork_rng(devices=[]):
    # Independent reference for seeded initialisation.
    torch.manual_seed(seed)
    reference_initial = CountingGRUSystem()

    # Exactly one optimisation update.
    trained, history, initial = train_counting(
        updates=1,
        seed=seed,
        data_seed=data_seed,
        val_seed=val_seed,
        device="cpu",
        batch_size=batch_size,
        log_every=1,
        return_initial=True,
    )

assert len(history) == 1
assert history[0]["update"] == 1

# Initial parameters match the independently seeded model.
for name, parameter in reference_initial.state_dict().items():
    assert torch.equal(initial.state_dict()[name], parameter)

assert all(not p.requires_grad for p in initial.parameters())

# At least one trainable parameter changed.
assert any(
    not torch.equal(initial.state_dict()[name], value)
    for name, value in trained.state_dict().items()
)

# Reconstruct the first minibatch with the specified data seed.
train_rng = torch.Generator().manual_seed(data_seed)
T = int(torch.randint(5, 21, (), generator=train_rng).item())

X, _, _ = generate_counting_batch(
    T, batch_size, p=0.5, generator=train_rng
)

# Independent Python integer accumulation for target counts.
target = torch.tensor([
    sum(int(X[t, b, 0].item()) for t in range(T))
    for b in range(batch_size)
], dtype=torch.float32)

with torch.no_grad():
    prediction = initial(X)
    expected_train_mse = (
        (prediction - target).square().mean().item()
    )

assert math.isclose(
    history[0]["train_mse"],
    expected_train_mse,
    rel_tol=1e-6,
    abs_tol=1e-5,
)

# Independently reconstruct fixed validation data.
val_rng = torch.Generator().manual_seed(val_seed)

total_squared_error = 0.0
n = 0

with torch.no_grad():
    for length in range(5, 21):
        Xv, _, _ = generate_counting_batch(
            length, 256, p=0.5, generator=val_rng
        )

        yv = torch.tensor([
            sum(int(Xv[t, b, 0].item()) for t in range(length))
            for b in range(256)
        ], dtype=torch.float32)

        error = trained(Xv) - yv
        total_squared_error += error.double().square().sum().item()
        n += yv.numel()

expected_val_mse = total_squared_error / n

assert n == 4096
assert math.isclose(
    history[0]["val_mse"],
    expected_val_mse,
    rel_tol=1e-6,
    abs_tol=1e-5,
)

# Analytical population benchmarks.
lengths = range(5, 21)
mean_T = sum(lengths) / len(lengths)
var_T = sum((T - mean_T)**2 for T in lengths) / len(lengths)

p = 0.5
constant_mse = p * (1 - p) * mean_T + p**2 * var_T
length_only_mse = p * (1 - p) * mean_T

assert math.isclose(constant_mse, 8.4375)
assert math.isclose(length_only_mse, 3.125)

print("train_counting verification passed.")
print(f"First sampled sequence length: {T}")
print(f"Training MSE: {history[0]['train_mse']:.6f}")
print(f"Validation MSE: {history[0]['val_mse']:.6f}")

{'update': 1, 'train_mse': 40.44905090332031, 'val_mse': 46.63703536987305}
train_counting verification passed.
First sampled sequence length: 13
Training MSE: 40.449051
Validation MSE: 46.637035


## 2. Evaluation — Level 1: Predictive Performance

The functions responsible for evaluating the counting GRU begin with the implementation of `regression_metrics` - lines 195–214.
It calculates four principal regression statistics and optionally a fifth metric measuring exact count accuracy after rounding.

#### 1.1. Mathematical definitions

Let the predictions, true counts, and signed errors for $N$ evaluated sequences be

$$
\hat{\boldsymbol y}=(\hat y_1,\ldots,\hat y_N),
\qquad
\boldsymbol y=(y_1,\ldots,y_N),
\qquad
e_i=\hat y_i-y_i.
$$

**Mean squared error (MSE)**

As defined in the Training section,

$$
\mathrm{MSE}=\frac1N\sum_{i=1}^{N}e_i^2.
$$

**Mean absolute error (MAE)**

$$
\boxed{\mathrm{MAE}=\frac1N\sum_{i=1}^{N}|e_i|.}
$$

Measures average prediction error in count units.

**Signed prediction bias**

$$
\boxed{\mathrm{Bias}=\frac1N\sum_{i=1}^{N}e_i.}
$$

Positive values indicate average overcounting; negative values indicate undercounting. Opposing errors can cancel, so zero bias does not establish accuracy.

**Coefficient of determination ($R^2$)**

$$
\boxed{
R^2=
1-\frac{\sum_i(\hat y_i-y_i)^2}
{\sum_i(y_i-\bar y)^2},
\qquad
\bar y=\frac1N\sum_i y_i.
}
$$

This compares squared prediction error against predicting the observed mean target. $R^2=1$ indicates perfect prediction; negative values are possible.

When all targets are identical, the denominator is zero and $R^2$ is undefined. The implementation correctly returns `None`. This is particularly relevant to fixed-count controls.

**Rounded exact-count accuracy**

$$
\boxed{
A_{\mathrm{round}}
=
\frac1N\sum_{i=1}^{N}
\mathbf1[\operatorname{round}(\hat y_i)=y_i].
}
$$

Measures the fraction of predicted counts that match the integer target after rounding. PyTorch uses ties-to-even rounding at exact half-integers.

Rounded accuracy complements continuous error metrics; it does not replace them.

#### 1.2. Implementation

The function:

1. Detaches prediction and target tensors from autograd.
2. Converts both to float64 and flattens them.
3. Calculates MSE, MAE, signed bias, and $R^2$.
4. Optionally calculates rounded accuracy.
5. Returns a dictionary containing the metrics and sample count `n`.

**Input mismatch** Predictions and targets correspond elementwise, contain equal nonzero numbers of elements, and contain finite numerical values. To ensure the current implementation does produce mismatched input sizes that may broadcast silently, in `regression_metrics` the following validation check is used after flattening the inputs:

`if prediction.numel() != target.numel() or target.numel() == 0:
    raise ValueError("Prediction and target must have equal, nonzero sizes.")`

This ensures that the metric function cannot silently compare one prediction with several targets through broadcasting. This is verified below

In [4]:
import math
import torch

from counting_experiment import regression_metrics

# Test A: Valid inputs
prediction = torch.tensor([1.2, 2.8, 4.1])
target = torch.tensor([1.0, 3.0, 4.0])

metrics = regression_metrics(prediction, target)

assert metrics["n"] == 3
print("Test A passed: valid inputs accepted.")

# Test B: Mismatched inputs
prediction = torch.tensor([2.0])
target = torch.tensor([1.0, 2.0, 3.0])

try:
    regression_metrics(prediction, target)
except ValueError:
    print("Test B passed: mismatched inputs rejected.")
else:
    raise AssertionError("Mismatched inputs were not rejected.")

# Test C: Empty inputs
empty = torch.empty(0)

try:
    regression_metrics(empty, empty)
except ValueError:
    print("Test C passed: empty inputs rejected.")
else:
    raise AssertionError("Empty inputs were not rejected.")


Test A passed: valid inputs accepted.
Test B passed: mismatched inputs rejected.
Test C passed: empty inputs rejected.



### 2. Predictive evaluation — `evaluate_predictive()`

The metrics from the previous section are applied across different sequence lengths and count distributions using the function `counting_experiment.py`, lines 217–249.

#### 2.1. Mathematical specification

For a batch containing $B$ binary sequences of common length $T$, the true count and model prediction for sequence $b$ are

$$
Y_b=\sum_{t=1}^{T}X_{t,b},
\qquad
\hat Y_b=f_\theta(X_{1:T}^{(b)}).
$$

Both are tensors of shape $(B,)$.

The function applies the metrics defined in the preceding `regression_metrics()` section to three evaluation outputs.

**A. Overall model performance**

The function calculates MSE, MAE, signed bias, $R^2$, and rounded accuracy across all sequences in the batch.

These measure continuous prediction quality and exact integer-count correctness.

**B. Length-only statistical reference**

The reference predictor is

$$
\boxed{
\hat Y_{\mathrm{length}}=pT.
}
$$

As established in the Data Generation and Training sections,

$$
\mathbb E[Y_T\mid T]=pT
$$

under iid Bernoulli($p$) sampling.

Therefore, $pT$ minimises expected squared error among predictors that observe sequence length but not the individual tokens.

Its theoretical conditional MSE is

$$
\boxed{
\mathcal R_{\mathrm{length}}(T)=Tp(1-p).
}
$$

For the default $p=0.5$,

$$
\hat Y_{\mathrm{length}}=T/2.
$$

**Important limitation:** This is a reference for the Bernoulli data-generating distribution, not an oracle for exact-count controls.

For sequences constrained to contain exactly $k$ ones, its MSE is

$$
\boxed{
\mathrm{MSE}_{pT\mid k}=(pT-k)^2.
}
$$

For example, when $T=40$, $k=6$, and $p=0.5$, the reference predicts 20 and has MSE

$$
(20-6)^2=196.
$$

This does not imply that the reference is incorrectly implemented; the evaluation distribution differs from the original Bernoulli distribution.

**C. Performance conditional on true count**

For each observed count $k$, define the subset

$$
\mathcal I_k=\{b:Y_b=k\}.
$$

The function evaluates the model separately on these observations.

For example, count-conditioned MSE is

$$
\boxed{
\mathrm{MSE}_k
=
\frac{1}{|\mathcal I_k|}
\sum_{b\in\mathcal I_k}(\hat Y_b-k)^2.
}
$$

The other metrics follow the definitions established in `regression_metrics()`.

This analysis can reveal count-specific failures concealed by aggregate statistics.

Because all targets within a count group are equal, $R^2$ is undefined and correctly returned as `None`.

Only counts actually observed in the evaluation batch appear in the report. The sample count `n` must therefore be considered when interpreting individual groups.

#### 2.2. Implementation and returned structure

For each labelled batch, the function:

1. Aligns input dtype and device with the model using `_model_input()`.
2. Extracts sequence length from `X.shape[0]`.
3. Reconstructs true counts using `X[..., 0].sum(0)`.
4. Computes model predictions without gradient tracking.
5. Constructs the length-only reference using `torch.full_like(target, p * T)`.
6. Computes overall and count-conditioned regression metrics.
7. Classifies the sequence length relative to `train_range`.

Each report contains:

| Key | Meaning |
|---|---|
| `T` | Sequence length |
| `length_region` | `ID_length` or `OOD_length` |
| `model` | Overall GRU prediction metrics |
| `length_only_pT` | Length-only reference metrics |
| `model_by_true_count` | Model metrics grouped by observed target count |

The default training-length support is

$$
\mathcal T_{\mathrm{ID}}=\{5,\ldots,20\}.
$$

Classification is defined by

$$
\mathrm{ID\_length}\iff 5\le T\le20.
$$

**Important distinction:** `ID_length` indicates that sequence length falls within the training-length support. It does not guarantee that the full token distribution or count composition is typical of training.

The `@torch.no_grad()` decorator prevents autograd recording during evaluation. The function does not itself call `model.eval()`; the supplied model is assumed to be in the intended evaluation mode. For the current GRU/readout architecture, there are no dropout or batch-normalisation layers.

#### 2.3. Important methodological comments

**Count-conditioned diagnostics**

Grouping by true count helps identify boundary failures or systematic dependence on count magnitude. Results from groups with few observations must be interpreted cautiously.

**Length-based ID/OOD classification**

The `length_region` field classifies only sequence length. Fixed-count conditions may be atypical under the training distribution even when their length is within the ID range.



## Level 2 — Hidden-State Representation Probes

**Source:** `counting_experiment.py`, functions `fit_ridge()`, `ridge_predict()`, `collect_probe_pairs()`, `fit_accumulator_probes()`, and `evaluate_probes()`.

**Central question:** Do the GRU's hidden states contain count-related information that can be recovered by linear decoders fitted exclusively on ID data, and does this decodability persist beyond the training horizon?

### 2.1. Representation dataset

For sequence $b$ at timestep $t$, let

$$
h_{t,b}\in\mathbb R^K,
\qquad
c_{t,b}=\sum_{s=1}^{t}x_{s,b},
\qquad
\tau_{t,b}=t.
$$

The function `collect_probe_pairs()` constructs observations

$$
\boxed{(h_{t,b},c_{t,b},t).}
$$

For a batch of $B$ sequences of length $T$, the resulting arrays have shapes

$$
H\in\mathbb R^{TB\times K},
\qquad
c,\tau\in\mathbb R^{TB}.
$$

Flattening preserves the time-major $(t,b)$ ordering. The elapsed-time vector uses `repeat_interleave(B)` to maintain alignment with hidden states and prefix counts.

Unlike the original model, which is trained on final counts, the probes receive supervision at every intermediate timestep.

### 2.2. Ridge regression

The function `fit_ridge()` fits a linear decoder

$$
\hat y=Aw+b
$$

by minimising

$$
\boxed{
\mathcal J(w,b)
=
\|Aw+b\mathbf1-y\|_2^2
+
\alpha\|w\|_2^2,
\qquad \alpha\geq0.
}
$$

The intercept is unpenalised.

After centring the features and targets, the regularised problem is represented as ordinary least squares:

$$
\boxed{
\begin{bmatrix}
A_c\\
\sqrt{\alpha}I
\end{bmatrix}w
\approx
\begin{bmatrix}
y_c\\
0
\end{bmatrix}.
}
$$

The intercept is recovered using

$$
b=\bar y-\bar a^\top w.
$$

The implementation uses an SVD-based least-squares solver on CPU float64 tensors, avoiding explicit inversion of the normal equations.

`ridge_predict()` evaluates the resulting affine map.

**Regularisation convention:** The objective penalises the sum, not the mean, of squared errors. Consequently, the effective strength of a fixed $\alpha$ depends on the number of fitting observations.

### 2.3. Standardisation and probe targets

`fit_accumulator_probes()` computes hidden-coordinate means and standard deviations using ID fitting data:

$$
z_{i,j}=\frac{h_{i,j}-\mu_j}{s_j}.
$$

For coordinates with standard deviation at most $10^{-12}$, the implementation substitutes $s_j=1$.

The same fitting-set statistics are subsequently used for validation and OOD evaluation.

Two linear probes are fitted.

**Direct count probe**

$$
\boxed{
\hat c_t^{(d)}=w_d^\top z_t+b_d.
}
$$

This attempts to decode the prefix count directly from hidden state.

**Residual count probe**

Define the residual relative to the Bernoulli expected count:

$$
\boxed{r_t=c_t-pt.}
$$

The residual probe predicts

$$
\hat r_t=w_r^\top z_t+b_r.
$$

A count prediction can be reconstructed using

$$
\boxed{
\hat c_t^{(r)}=pt+\hat r_t.
}
$$

**Critical distinction:** The reconstructed residual probe receives elapsed time $t$ externally. The direct probe does not. These are therefore different information conditions, not strictly equivalent tests of the hidden representation alone.

### 2.4. ID calibration and frozen OOD evaluation

Regularisation is selected separately for the two probes using independent ID validation data:

$$
\boxed{
\alpha_j^*
=
\arg\min_{\alpha\in\mathcal A}
\mathrm{MSE}_{\mathrm{val},j}(\alpha),
\qquad j\in\{d,r\}.
}
$$

The default candidate grid is

$$
\mathcal A=\{10^{-6},10^{-3},1,1000\}.
$$

Ties retain the first candidate.

Neither probe coefficients nor feature-standardisation statistics are refitted on OOD data.

`evaluate_probes()` reports:

| Comparison | Prediction | Target |
|---|---|---|
| Direct probe | $\hat c_t^{(d)}$ | $c_t$ |
| Time-only reference | $pt$ | $c_t$ |
| Residual probe | $\hat r_t$ | $c_t-pt$ |
| Zero-residual reference | $0$ | $c_t-pt$ |
| Reconstructed count | $pt+\hat r_t$ | $c_t$ |

The residual error satisfies

$$
(pt+\hat r_t)-c_t=\hat r_t-(c_t-pt).
$$

Therefore, on identical observations,

$$
\boxed{
\mathrm{MSE}_{\mathrm{reconstructed}}
=
\mathrm{MSE}_{\mathrm{residual}}.
}
$$

The same identity holds for MAE and signed bias, up to numerical precision.

Evaluation is partitioned into:

- `all_steps`: every evaluated prefix;
- `steps_beyond_training_horizon`: timesteps $t>T_{\max}^{\mathrm{train}}$;
- `by_elapsed_time`: selected individual timesteps.

An OOD-length sequence may contain many prefixes within the original training horizon. The beyond-horizon subset isolates the later recurrent transitions.

### 2.5. Assumptions and methodological comments

**Data separation.** Probe fitting and validation must use independently sampled ID data. The idea is that OOD evaluation must not influence standardisation or regularisation selection. The code checks length eligibility; sample independence and distribution provenance remain protocol requirements.

**Observation weighting.** Each timestep contributes a probe observation. With equal numbers of sequences per length, longer sequences contribute more observations. This differs from the length-balanced final-count training objective.

**Dependent observations.** Multiple prefix observations from the same sequence are statistically dependent. The number of timestep observations is not the number of independent sequences.

**Decodability versus mechanism.** A successful linear probe establishes that a fitted linear map can recover count-related information from the observed hidden states. It does not establish that the GRU's own readout uses this map or that the hidden state implements an exact accumulator.

**OOD interpretation.** Deteriorating probe performance could reflect representational changes, poor linear extrapolation, or evaluation-distribution differences. *Identifying the cause of prediction failure is a very important question that we want to probe using the vanilla GRU.*


In [2]:

import math

import torch

from counting_experiment import (
    CountingGRUSystem,
    collect_probe_pairs,
    evaluate_probes,
    fit_accumulator_probes,
    fit_ridge,
    length_batches,
    ridge_predict,
)

# 1. Ridge regression with known analytical solution.
A = torch.tensor([[0.], [1.], [2.]], dtype=torch.float64)
y = torch.tensor([1., 3., 5.], dtype=torch.float64)

weight, bias = fit_ridge(A, y, alpha=3.0)

torch.testing.assert_close(weight, torch.tensor([0.8], dtype=torch.float64))
torch.testing.assert_close(bias, torch.tensor(2.2, dtype=torch.float64))

# 2. Verify time-major alignment of counts and elapsed times.
model = CountingGRUSystem().eval()

X = torch.tensor([
    [1., 0.],
    [0., 1.],
    [1., 0.],
])[..., None]

H, counts, times = collect_probe_pairs(model, {"test": X})

assert H.shape == (6, model.gru.num_hiddens)
torch.testing.assert_close(
    counts, torch.tensor([1., 0., 1., 1., 2., 1.], dtype=torch.float64)
)
torch.testing.assert_close(
    times, torch.tensor([1., 1., 2., 2., 3., 3.], dtype=torch.float64)
)

# 3. Fit on ID data and evaluate frozen probes beyond the horizon.
fitting = length_batches((5, 6), batch_size=4, seed=74)
validation = length_batches((5, 6), batch_size=4, seed=75)

probes = fit_accumulator_probes(
    model,
    fitting,
    alpha=(1e-3, 1.0),
    validation_batches=validation,
)

mean_before = probes["feature_mean"].clone()
scale_before = probes["feature_scale"].clone()

ood = length_batches((21,), batch_size=4, seed=76)
report = evaluate_probes(model, probes, ood)[21]

# Reconstructed count and residual have identical errors.
for subset in ("all_steps", "steps_beyond_training_horizon"):
    metrics = report[subset]
    for name in ("mse", "mae", "bias"):
        assert math.isclose(
            metrics["hidden_to_count_residual"][name],
            metrics["time_plus_hidden_residual"][name],
            abs_tol=1e-10,
        )

# Evaluation must not modify fitting-set standardisation.
assert torch.equal(probes["feature_mean"], mean_before)
assert torch.equal(probes["feature_scale"], scale_before)

print("Level 2 verification passed.")


Level 2 verification passed.



## Level 3 — Gate Interventions and Local Recurrent Dynamics

**Source:** `counting_experiment.py`, functions `clamp_z_on_zero()`, `identity_z()`, `gate_statistics()`, `compare_gate_intervention()`, `zero_tail_drift()`, `origin_jacobian()`, `recurrent_step()`, and `dynamical_diagnostics()`.

**Central question:** How do update-gate interventions affect counting behaviour, and what do the local recurrent dynamics reveal about the preservation or amplification of hidden-state perturbations?

The motivation for Level 3 consists of two complementary analyses:

1. **Gate interventions:** We want to deliberately modify recurrent transitions and observe their consequences.
2. **Local dynamics:** We want to measure hidden-state sensitivity using Jacobians and their products.

The GRU recurrence and parameter conventions are established in the architecture section and are not rederived here.

### 3.1. Zero-token update-gate intervention

The function `clamp_z_on_zero()` modifies the update gate according to

$$
\boxed{
Z_t^{\mathrm{effective}}=
\begin{cases}
\mathbf1,&x_t=0,\\
Z_t,&x_t=1.
\end{cases}
}
$$

Under the established GRU update convention,

$$
H_t=Z_t\odot H_{t-1}+(1-Z_t)\odot\widetilde H_t.
$$

For a zero token, setting $Z_t=\mathbf1$ gives

$$
\boxed{H_t=H_{t-1}.}
$$

Therefore, the intervention forces exact hidden-state preservation during zero-token transitions.

The function `identity_z()` returns the computed gate unchanged and provides a negative control:

$$
\boxed{
f_\theta(X;\mathrm{identity})=f_\theta(X;\mathrm{ordinary}).
}
$$

This checks that the override mechanism does not alter predictions when no actual gate modification is requested.

**Interpretation:** Hidden-state preservation is imposed by the intervention. Its occurrence under clamping is not evidence that the unmodified model learned this behaviour.

### 3.2. Gate statistics and paired interventions

`gate_statistics()` summarises recorded gate values for all token positions, zero-token positions, and one-token positions.

For gate coordinate $j$ and group $\mathcal G$, the per-unit mean is

$$
\boxed{
\bar g_{\mathcal G,j}
=
\frac{1}{|\mathcal G|}
\sum_{(t,b)\in\mathcal G}g_{t,b,j}.
}
$$

The function also reports pooled means, standard deviations, quantiles, and position counts.

Pooled statistics may conceal substantial differences between hidden units; per-unit summaries should therefore be retained.

`compare_gate_intervention()` evaluates three conditions on the same sequences:

| Condition | Description |
|---|---|
| `ordinary` | Unmodified recurrent computation |
| `clamped` | Force $Z_t=1$ at zero tokens |
| `matched_random` | Force $Z_t=1$ at a matched number of randomly selected positions |

For sequence $b$, let $m_b$ denote the number of zero tokens. The random control selects exactly $m_b$ distinct positions without conditioning on token identity, apart from the selection count.

If the sequence contains $k_b$ ones, then

$$
\boxed{
\mathbb E[N_{\mathrm{ones\ clamped}}\mid X]
=
m_b\frac{k_b}{T}.
}
$$

Thus, the random intervention may suppress transitions involving one tokens, potentially preventing legitimate count increments.

**Important limitation:** The two interventions match the number of affected positions, not their semantic role or the magnitude of gate changes. Differences between them do not isolate zero-token leakage as a unique causal mechanism.

The function additionally records predictions, intermediate states, state norms, gate summaries, and exact-count metrics.

Intermediate readout predictions are diagnostic: the original GRU was supervised on final counts rather than every prefix.

### 3.3. Zero-tail stability

`zero_tail_drift()` starts from the hidden state reached after processing an ordinary prefix.

It then appends $m$ zero tokens and measures changes in the final prediction and hidden state.

Let

$$
h_T=F_\theta(x_{1:T}),
\qquad
\hat y_T=W_{\mathrm{out}}h_T+b_{\mathrm{out}}.
$$

After appending $m$ zeros, define

$$
\boxed{
\Delta_m=\hat y_{T+m}-\hat y_T
}
$$

and

$$
\boxed{
d_m=\|h_{T+m}-h_T\|_2.
}
$$

The function reports mean signed drift, mean absolute drift, hidden displacement, and final count-prediction errors.

For the clamped intervention, every appended zero transition preserves the hidden state. Consequently,

$$
\boxed{
h_{T+m}^{\mathrm{clamped}}=h_T,
\qquad
\Delta_m^{\mathrm{clamped}}=0,
\qquad
d_m^{\mathrm{clamped}}=0.
}
$$

These are analytical invariants of the intervention.

**Important distinction:** Zero prediction drift does not establish correct counting. A model can preserve an already incorrect prediction. The function appropriately reports count error separately.

### 3.4. Local recurrent Jacobian

Let the single-step transition in column-vector notation be

$$
h_t=F_\theta(x_t,h_{t-1}).
$$

The local Jacobian is

$$
\boxed{
J_t=
\frac{\partial F_\theta(x_t,h)}
{\partial h}
\bigg|_{h=h_{t-1}}
\in\mathbb R^{K\times K}.
}
$$

It measures the first-order response of the next hidden state to perturbations of the previous hidden state, holding the input fixed.

`recurrent_step()` exposes one GRU transition so that this derivative can be calculated directly.

`dynamical_diagnostics()` evaluates Jacobians along a trajectory beginning at $h_0=0$, using `torch.func.jacrev`.

For each selected transition, it reports:

**Spectral radius**

$$
\boxed{
\rho(J_t)=\max_i|\lambda_i(J_t)|.
}
$$

**Spectral norm**

$$
\boxed{
\|J_t\|_2=
\max_{\|v\|_2=1}\|J_tv\|_2.
}
$$

These quantities are distinct. For non-normal Jacobians, a spectral radius below one does not preclude transient perturbation amplification.

### 3.5. Analytical origin Jacobian

At $x=0$, $h=0$, and with all GRU bias vectors equal to zero, the update and reset gates satisfy

$$
Z=R=\frac12\mathbf1.
$$

The exact column-convention Jacobian is

$$
\boxed{
J_{\mathrm{origin}}
=
\frac12I_K+\frac14W_{hh}^{\top}.
}
$$

For the implementation's row-vector convention, the corresponding perturbation map is

$$
\boxed{
M_{\mathrm{row}}
=
\frac12I_K+\frac14W_{hh}.
}
$$

`origin_jacobian()` correctly returns both expressions and rejects nonzero biases.

This formula is a special-case analytical reference. It must not be substituted for trajectory Jacobians when the origin assumptions do not hold.

### 3.6. Multistep sensitivity

For successive recurrent transitions, the chain rule gives

$$
\boxed{
\frac{\partial h_b}{\partial h_{a-1}}
=
J_bJ_{b-1}\cdots J_a.
}
$$

The ordering is essential.

`dynamical_diagnostics()` constructs the corresponding products and reports spectral norms of successive prefix products.

In its Python interface, `steps` use zero-based transition indices and `windows` use half-open intervals `(start, stop)`.

For a window `(0, 2)`, the complete product is

$$
\boxed{P=J_2J_1.}
$$

The function calculates only the selected local Jacobians and those required for requested windows. Small windows are appropriate for focused analysis.

**Interpretation:** Jacobians characterise infinitesimal state sensitivity around an observed trajectory. They do not directly measure count accuracy, nor do local norms uniquely determine long-horizon behaviour.

### 3.7. Assumptions and methodological findings

**Intervention semantics.** Under the GRU convention employed, $Z=1$ preserves the previous state. This should be percieved as *full retention*, not ambiguously as a fully open gate.

**Counterfactual dependence.** Changing a gate affects the subsequent hidden trajectory and therefore later computed gates. Final prediction differences reflect the full intervened trajectory, not necessarily one isolated transition.

**Matched controls.** A matched number of modified positions does not imply matched causal impact.

**Stability versus accuracy.** Stable hidden states or predictions can preserve an incorrect count.

**Local versus global dynamics.** Jacobian norms describe first-order sensitivity at particular states. They do not establish the effect of arbitrary finite perturbations or identify an exact accumulator mechanism.

**Mechanistic inference.** Level 3 supplies controlled perturbation evidence and dynamical diagnostics. Any claim about a learned counting mechanism requires consistency with the Level 1 predictive results and Level 2 representation findings.

### 3.8. Methodological comments

Gate interventions and local Jacobians examine different aspects of the same recurrent system. The idea is to utilise **gate interventions** to test the effects of specified changes to recurrent computation; while we use Jacobians to probe local sensitivity of that computation. In principle, both may help explain observed counting behaviour.

The overarching intertest, of course, it to try and establish that the model learned an *exact counting algorithm*. The experimental set-up is only one piece of an exploratory hypothesis.
